In [1]:
# 05_merge_final_table
#
# 목적: 01~04단계 산출물을 msno 기준으로 병합해 최종 모델링 테이블을 만든다.
#      결측 대체(중앙값 등)나 스케일링 같은 "통계 기반 정제"는 여기서 하지 않는다
#      (train split으로만 fit해야 하므로 모델링 직전 별도 단계에서 처리, plan 문서 참고).
#      이 노트북은 순수 병합 + 커버리지(결측) 확인까지만 수행한다.
#
# 입력:
#   data/processed/labels_pooled.csv       (msno, snapshot, is_churn)
#   data/processed/user_split.csv          (msno, split)
#   data/processed/features_members.csv
#   data/processed/features_transactions.csv
#   data/processed/features_user_logs.csv
# 출력:
#   data/processed/model_table.csv         (msno, snapshot, split, is_churn, + 모든 피처)

In [2]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

labels = pd.read_csv(PROCESSED_DIR / "labels_pooled.csv")
user_split = pd.read_csv(PROCESSED_DIR / "user_split.csv")
features_members = pd.read_csv(PROCESSED_DIR / "features_members.csv")
features_transactions = pd.read_csv(PROCESSED_DIR / "features_transactions.csv")
features_user_logs = pd.read_csv(PROCESSED_DIR / "features_user_logs.csv")

print("labels:", labels.shape)
print("user_split:", user_split.shape)
print("features_members:", features_members.shape)
print("features_transactions:", features_transactions.shape)
print("features_user_logs:", features_user_logs.shape)

labels: (1963891, 3)
user_split: (1082190, 2)
features_members: (6769473, 7)
features_transactions: (2363626, 15)
features_user_logs: (5234111, 18)


In [3]:
table = labels.merge(user_split, on="msno", how="left")
assert table["split"].isna().sum() == 0, "split 미배정 유저 존재"

table = table.merge(features_members, on="msno", how="left")
table = table.merge(features_transactions, on="msno", how="left")
table = table.merge(features_user_logs, on="msno", how="left")

print(table.shape)
table.head()

(1963891, 41)


,msno,snapshot,is_churn,split,city,registered_via,bd_clean,bd_is_missing,gender,tenure_days,...,d30_completion_rate,d90_active_days,d90_avg_num_unq,d90_avg_total_secs,d90_completion_rate,full_active_days,full_avg_num_unq,full_avg_total_secs,full_completion_rate,trend_secs_recent15_vs_prior15
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,2017-02,1,valid,18.0,9.0,36.0,0.0,female,4346.0,...,0.746835,24.0,18.208333,4310.198583,0.808732,26.0,17.769231,4155.840538,0.808300,NaN
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,2017-02,1,train,10.0,9.0,38.0,0.0,male,4345.0,...,1.000000,7.0,42.000000,14805.402286,0.926591,521.0,13.880998,5019.792971,0.916736,NaN
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,2017-02,1,train,11.0,9.0,27.0,0.0,female,4153.0,...,0.250000,15.0,37.333333,6405.242800,0.614238,237.0,48.662447,10959.428928,0.752437,NaN
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,2017-02,1,valid,13.0,9.0,23.0,0.0,female,4136.0,...,0.505698,66.0,19.727273,4353.639439,0.483855,735.0,25.141497,6728.311370,0.579575,0.466362
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,2017-02,1,train,3.0,9.0,27.0,0.0,male,4080.0,...,0.931424,70.0,67.957143,17355.256314,0.954194,758.0,95.875989,24596.500005,0.919552,-0.224411


In [4]:
# 커버리지(결측) 확인: 각 소스 테이블에 매칭되지 않은 유저 비율
members_missing = table["city"].isna().mean()
txn_missing = table["txn_count"].isna().mean()
logs_missing = table["full_active_days"].isna().mean()

print(f"members_v3에 없는 유저 비율: {members_missing * 100:.2f}%")
print(f"transactions에 없는 유저 비율: {txn_missing * 100:.2f}%")
print(f"user_logs에 없는 유저 비율: {logs_missing * 100:.2f}%")

members_v3에 없는 유저 비율: 11.50%
transactions에 없는 유저 비율: 0.13%
user_logs에 없는 유저 비율: 12.41%


In [5]:
# 그룹 무결성 재검증 + split별 요약 (01단계 결과와 일치해야 함)
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    users_a = set(table.loc[table["split"] == a, "msno"])
    users_b = set(table.loc[table["split"] == b, "msno"])
    assert len(users_a & users_b) == 0, f"{a}/{b} 간 유저 중복 발견"
print("그룹 무결성 재검증 통과")

table.groupby("split").agg(rows=("msno", "size"), users=("msno", "nunique"), churn_rate=("is_churn", "mean"))

그룹 무결성 재검증 통과


,rows,users,churn_rate
split,,,
test,294477,162329,0.076773
train,1374692,757533,0.076862
valid,294722,162328,0.076448


In [6]:
print("컬럼 목록:")
print(table.columns.tolist())
print()
print("전체 결측치 비율 (상위):")
print((table.isna().mean() * 100).round(2).sort_values(ascending=False).head(15))

컬럼 목록:
['msno', 'snapshot', 'is_churn', 'split', 'city', 'registered_via', 'bd_clean', 'bd_is_missing', 'gender', 'tenure_days', 'txn_count', 'payment_method_nunique', 'plan_days_nunique', 'auto_renew_rate', 'cancel_count', 'cancel_rate', 'total_amount_paid', 'avg_discount', 'last_payment_plan_days', 'last_plan_list_price', 'last_actual_amount_paid', 'last_is_auto_renew', 'days_since_last_txn', 'days_to_expire', 'd7_active_days', 'd7_avg_num_unq', 'd7_avg_total_secs', 'd7_completion_rate', 'd30_active_days', 'd30_avg_num_unq', 'd30_avg_total_secs', 'd30_completion_rate', 'd90_active_days', 'd90_avg_num_unq', 'd90_avg_total_secs', 'd90_completion_rate', 'full_active_days', 'full_avg_num_unq', 'full_avg_total_secs', 'full_completion_rate', 'trend_secs_recent15_vs_prior15']

전체 결측치 비율 (상위):


bd_clean                          60.49
d7_completion_rate                31.30
d7_avg_num_unq                    31.30
d7_avg_total_secs                 31.30
trend_secs_recent15_vs_prior15    30.13
d30_avg_num_unq                   22.29
d30_completion_rate               22.29
d30_avg_total_secs                22.29
d90_avg_num_unq                   18.38
d90_avg_total_secs                18.38
d90_completion_rate               18.38
d7_active_days                    12.41
full_avg_total_secs               12.41
d30_active_days                   12.41
d90_active_days                   12.41
dtype: float64


In [7]:
table.to_csv(PROCESSED_DIR / "model_table.csv", index=False)
print(f"저장 완료: {PROCESSED_DIR / 'model_table.csv'} ({len(table):,} rows, {len(table.columns)} columns)")

저장 완료: ..\data\processed\model_table.csv (1,963,891 rows, 41 columns)
